In [ ]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup

# Nutrient pages to scrape
nutrients = {
    "Vitamin C": "VitaminC",
    "Vitamin A": "VitaminA",
    "Vitamin D": "VitaminD",
    "Vitamin E": "VitaminE",
    "Vitamin B6": "VitaminB6",
    "Vitamin B12": "VitaminB12",
    "Calcium": "Calcium",
    "Iron": "Iron",
    "Zinc": "Zinc",
    "Magnesium": "Magnesium",
}

BASE_URL = "https://ods.od.nih.gov/factsheets/{}/HealthProfessional/"

# Correct NIH fact-sheet URL format
BASE_URL = "https://ods.od.nih.gov/factsheets/{}-HealthProfessional/"

session = requests.Session()
session.headers.update({
    "User-Agent": "NutritionResearchProject/1.0"
})

def extract_number(value):
    """Convert values such as '2,000 mg' to 2000."""
    match = re.search(r"[\d,]+(?:\.\d+)?", str(value))
    if match:
        return float(match.group().replace(",", ""))
    return None


def find_table(soup, keyword):
    """Find a table based on its caption or preceding heading."""
    for table in soup.find_all("table"):
        caption = table.find("caption")
        caption_text = caption.get_text(" ", strip=True) if caption else ""

        previous = table.find_previous(["h2", "h3", "h4", "p"])
        previous_text = previous.get_text(" ", strip=True) if previous else ""

        if keyword.lower() in (caption_text + " " + previous_text).lower():
            return table

    return None


def extract_adult_values(table):
    """Extract values for adults aged 19+."""
    if table is None:
        return {}

    for tr in table.find_all("tr"):
        cells = [
            cell.get_text(" ", strip=True)
            for cell in tr.find_all(["th", "td"])
        ]

        if len(cells) < 3:
            continue

        age = cells[0].lower().replace("–", "-")

        if "19+" in age or "19 years and older" in age:
            return {
                "Male": extract_number(cells[1]),
                "Female": extract_number(cells[2])
            }

    return {}


records = []

for nutrient, slug in nutrients.items():

    url = BASE_URL.format(slug)
    print("Scraping:", url)

    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # RDA table
        rda_table = find_table(
            soup, "Recommended Dietary Allowances"
        )

        # Upper limit table
        ul_table = find_table(
            soup, "Tolerable Upper Intake Levels"
        )

        rda = extract_adult_values(rda_table)
        ul = extract_adult_values(ul_table)

        # Skip pages where neither table was found
        if not rda and not ul:
            print("No matching tables:", nutrient)
            continue

        for gender in ["Male", "Female"]:

            records.append({
                "disease": "Healthy",
                "nutrient": nutrient,
                "min-value": rda.get(gender),
                "max-value": ul.get(gender),
                "unit": None,
                "age-group": "Adult",
                "gender": gender
            })

    except requests.RequestException as e:
        print("Error:", nutrient, e)


df = pd.DataFrame(records)

df.to_csv("nih_nutrient_dataset.csv", index=False)

print(df)
print("Total rows:", len(df))